In [30]:
import numpy as np
import os #allows python to interact with the operating system
os.environ["KERAS_BACKEND"] = "tensorflow" # Tell Keras to use TF as its engine
import keras
# source Gradioexp/bin/activate  for terminal activation

MNIST is a classic dataset of 70,000 handwritten digit images (0–9), each being a 28×28 grayscale image.

# Data Loading & Preprocessing

In [31]:
#1) Load the data and split it between train and test sets

#keras.datasets.mnist.load_data():Built-in Keras utility function that automatically downloads, caches, and loads the MNIST dataset directly from the internet — no manual downloading needed.
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

#70,000 total
#60,000 → Training (85.7%)
#10,000 → Testing  (14.3%)

#2) Normalize (Scale images to the [0, 1] range)
 
# Raw pixel values are integers from 0 to 255
# Dividing by 255 squishes them to 0.0 to 1.0

#Neural networks train faster and more stably with small input values.
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

#3) Reshape to make data ready to feed into a CNN
#CNNs (Convolutional Neural Networks) expect input as (batch, height, width, channels)
# Make sure images have shape (28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape) #one-dimensional array
print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")


x_train shape: (60000, 28, 28, 1)
y_train shape: (60000,)
60000 train samples
10000 test samples


In [32]:
# Model parameters
num_classes = 10
input_shape = (28, 28, 1)

What is keras.Sequential?
It's a linear stack of layers where data flows from top to bottom, one layer at a time.

In [33]:
model = keras.Sequential(
    [
        keras.layers.Input(shape=input_shape), #(28, 28, 1)
        keras.layers.Flatten(), #Unrolls the 2D image into a 1D vector (28*28*1 = 784)
        keras.layers.Dense(256, activation="relu"), #learn features
        #256 neurons, each connected to all 784 inputs
        #784 inputs × 256 neurons = 200,704 connections
        keras.layers.Dropout(0.2),#randomly switch OFF 20% of neurons during training to prevent overfitting
        keras.layers.Dense(128, activation="relu"),#Learns higher-level combinations of patterns from layer above
        keras.layers.Dropout(0.2),
        keras.layers.Dense(num_classes, activation="softmax"),#softmax converts raw scores into probabilities that sum to 1.0

    ]
)

# Neural Network Architecture
```
   Input Image (28×28×1)
        │
        ▼
   Flatten → 784 values
        │
        ▼
   Dense(256) + ReLU → 256 feature signals
        │
        ▼
   Dropout(0.2) → ~205 active neurons
        │
        ▼
   Dense(128) + ReLU → 128 abstract features
        │
        ▼
   Dropout(0.2) → ~102 active neurons
        │
        ▼
   Dense(10) + Softmax → [0.01, 0.02, 0.91, ...]
                                        ▲
                                  Final prediction
```


In [34]:
# model = keras.Sequential(
#     [
#         keras.layers.Input(shape=input_shape),
#         keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
#         keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
#         keras.layers.MaxPooling2D(pool_size=(2, 2)),
#         keras.layers.Conv2D(128, kernel_size=(3, 3), activation="relu"),
#         keras.layers.Conv2D(128, kernel_size=(3, 3), activation="relu"),
#         keras.layers.GlobalAveragePooling2D(),
#         keras.layers.Dropout(0.5),
#         keras.layers.Dense(num_classes, activation="softmax"),
#     ]
# )


In [35]:
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 235,146 (918.54 KB)

 Trainable params: 235,146 (918.54 KB)

 Non-trainable params: 0 (0.00 B)

```
(input_features × neurons) + biases
(784 × 256) + 256
= 200,704 + 256
= 200,960 
```

In [36]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(), 
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name="acc"),
    ],
)

#updates the weights to reduce the error (loss)
#Adam = Adaptive Moment Estimation(each weight has its own learning rate that adapts during training)


In [37]:
batch_size = 128
epochs = 20

callbacks = [
    #keras.callbacks.ModelCheckpoint(filepath="model_at_epoch_{epoch}.keras"),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3),
]

model.fit(
    x_train,
    y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.15,
    callbacks=callbacks,
)
score = model.evaluate(x_test, y_test, verbose=0)


Epoch 1/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - acc: 0.8910 - loss: 0.3679 - val_acc: 0.9620 - val_loss: 0.1321
Epoch 2/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - acc: 0.9534 - loss: 0.1559 - val_acc: 0.9690 - val_loss: 0.1021
Epoch 3/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - acc: 0.9664 - loss: 0.1104 - val_acc: 0.9749 - val_loss: 0.0855
Epoch 4/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - acc: 0.9733 - loss: 0.0859 - val_acc: 0.9768 - val_loss: 0.0743
Epoch 5/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - acc: 0.9769 - loss: 0.0723 - val_acc: 0.9790 - val_loss: 0.0729
Epoch 6/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - acc: 0.9815 - loss: 0.0592 - val_acc: 0.9777 - val_loss: 0.0768
Epoch 7/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - acc: 0.9834 - loss: 0.0516 - val_acc: 0.9776 - val_loss: 0.0746
Epoch 8/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - acc: 0.9851 - loss: 0.0465 - val_acc: 0.9792 - val_loss: 0.0730


In [38]:
model.save("final_model.keras")


In [39]:
model = keras.saving.load_model("final_model.keras")


In [40]:
predictions = model.predict(x_test)


313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step


In [41]:
y_pred = np.argmax(predictions, axis=1)

In [42]:
y_test

array([7, 2, 1, ..., 4, 5, 6], shape=(10000,), dtype=uint8)

In [43]:
y_pred

array([7, 2, 1, ..., 4, 5, 6], shape=(10000,))

In [44]:
accuracy = np.mean(y_pred == y_test)
print(f"Accuracy: {accuracy*100.:.4f}")

Accuracy: 98.1400


In [45]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred, digits=4))
cm = confusion_matrix(y_test, y_pred)

              precision    recall  f1-score   support

           0     0.9877    0.9867    0.9872       980
           1     0.9903    0.9921    0.9912      1135
           2     0.9825    0.9797    0.9811      1032
           3     0.9821    0.9772    0.9797      1010
           4     0.9797    0.9837    0.9817       982
           5     0.9691    0.9832    0.9761       892
           6     0.9803    0.9875    0.9839       958
           7     0.9834    0.9776    0.9805      1028
           8     0.9824    0.9743    0.9784       974
           9     0.9742    0.9713    0.9727      1009

    accuracy                         0.9814     10000
   macro avg     0.9812    0.9813    0.9812     10000
weighted avg     0.9814    0.9814    0.9814     10000



In [46]:
print(cm)


[[ 967    1    0    1    1    2    4    1    2    1]
 [   0 1126    3    1    0    1    2    0    2    0]
 [   4    2 1011    1    1    0    2    5    5    1]
 [   0    0    1  987    0   13    0    3    3    3]
 [   2    1    2    0  966    0    5    1    0    5]
 [   2    0    0    3    1  877    5    0    1    3]
 [   1    2    0    1    3    3  946    0    2    0]
 [   1    2    8    3    0    0    0 1005    1    8]
 [   0    1    4    4    2    7    1    1  949    5]
 [   2    2    0    4   12    2    0    6    1  980]]
